# YOLO Pith Detection with Controlled Full-Board and Tiled Experiments

This notebook trains native Ultralytics PyTorch detection models for Pith. Inference is performed independently for each Top or Bot TIFF image. The test split is locked after the persisted grouped split is created. No model export other than native `.pt` checkpoints is permitted.

Expensive cells are marked explicitly. Audit and geometry cells are mandatory; tiled generation, Optuna, final training, and locked-test evaluation are restart-safe and refuse to overwrite existing artifacts unless the corresponding force flag is enabled.

In [ ]:
from __future__ import annotations

import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from functions.yolo_detection import (
    box_iou, clip_box_to_tile, file_sha256, finite_or_error, parse_image_identity,
    percentile_summary, read_tiff_metadata, restore_tile_box, tile_windows,
    write_json, xywhn_to_xyxy, xyxy_to_xywhn,
)


def supported_kwargs(callable_object, requested):
    parameters = inspect.signature(callable_object).parameters
    if any(parameter.kind == inspect.Parameter.VAR_KEYWORD for parameter in parameters.values()):
        return dict(requested)
    unsupported = sorted(set(requested) - set(parameters))
    if unsupported:
        raise RuntimeError(f'Installed Ultralytics API does not support arguments: {unsupported}')
    return dict(requested)

try:
    import yaml
except ImportError as exc:
    raise RuntimeError('PyYAML is required to write the Ultralytics dataset YAML.') from exc

try:
    from ultralytics import YOLO, __version__ as ULTRALYTICS_VERSION
except ImportError as exc:
    raise RuntimeError('Ultralytics is required. Install the repository-pinned package before continuing.') from exc

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Ultralytics:', ULTRALYTICS_VERSION)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {props.name}, VRAM={props.total_memory / 2**30:.1f} GiB')

for method in ('train', 'val', 'predict'):
    signature = inspect.signature(getattr(YOLO, method))
    print(f'YOLO.{method}: {signature}')

In [ ]:
# Central configuration. Change values here only.
DATASET_ROOT = PROJECT_ROOT / 'data' / 'bbox_clean_yolo'
IMAGE_DIR = DATASET_ROOT / 'images'
LABEL_DIR = DATASET_ROOT / 'labels'
CLASS_MAPPING_PATH = DATASET_ROOT / 'class_mapping.json'
DATASET_MANIFEST_PATH = DATASET_ROOT / 'dataset_manifest.json'
ARTIFACT_ROOT = PROJECT_ROOT / 'data' / 'yolo_pith_detection'
SPLIT_MANIFEST_PATH = ARTIFACT_ROOT / 'split_manifest.json'
YOLO_DATASET_ROOT = ARTIFACT_ROOT / 'yolo_dataset'
TILE_DATASET_ROOT = ARTIFACT_ROOT / 'tiles'
DATA_YAML_PATH = YOLO_DATASET_ROOT / 'data.yaml'
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = "YOLO_Pith_Detection_20260925_174146"    #f'YOLO_Pith_Detection_{RUN_TIMESTAMP}'
OPTUNA_STUDY_NAME = 'YOLO_Pith_Detection_tiled_1024x768'

# Select the native pretrained Ultralytics detection checkpoint here.
PRETRAINED_DETECTION_CHECKPOINT = 'yolo26n.pt'

MODEL_ROOT = PROJECT_ROOT / 'models' / RUN_NAME
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / RUN_NAME
LOG_ROOT = PROJECT_ROOT / 'logs' / 'tensorboard' / RUN_NAME
OPTUNA_DB_PATH = PROJECT_ROOT / 'logs' / 'optuna' / 'YOLO_Pith_Detection.db'
METRICS_ROOT = PROJECT_ROOT / 'data' / 'metrics' / RUN_NAME
REVIEW_ROOT = ARTIFACT_ROOT / 'prediction_review' / RUN_NAME
DEPLOYMENT_CONFIG_PATH = MODEL_ROOT / 'deployment_config.json'

# Runtime and reproducibility.
SEED = 42
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DETERMINISTIC = False
NUM_WORKERS = 2
CACHE = False
AMP = True

# Dataset, split, and label policy.
GROUP_KEY = 'leading_numeric_id'
SPLIT_ALGORITHM_VERSION = 2
SPLIT_FRACTIONS = {'train': 0.64, 'val': 0.16, 'test': 0.20}
MODE = 'tiled'  # 'full' or 'tiled'
if MODE not in {'full', 'tiled'}:
    raise ValueError("MODE must be either 'full' or 'tiled'")
EMPTY_LABEL_POLICY = 'verified_negative_from_manifest_and_empty_file'
MISSING_LABEL_POLICY = 'fail'

# Full-board and tile geometry.
FULL_BOARD_IMGSZ_CANDIDATES = [1024]
TRAIN_IMGSZ = 1024
RECT = True
TILE_WIDTH = 1024
TILE_HEIGHT = 768
TILE_OVERLAP = 0.25
MIN_RETAINED_BOX_FRACTION = 0.25
MIN_RETAINED_BOX_WIDTH = 8
MIN_RETAINED_BOX_HEIGHT = 4
NEGATIVE_TILE_RATIO = 1.0
MERGE_METHOD = 'nms'
MERGE_IOU = 0.50

# Training and optimization.
BATCH = 8
EPOCHS = 150
TRIAL_EPOCHS = 50
PATIENCE = 20
SAVE_PERIOD = 10
N_TRIALS = 0
CONFIDENCE_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
NMS_IOU_GRID = np.round(np.arange(0.30, 0.91, 0.05), 2)
MIN_RECALL = 0.90
MAX_FP_PER_NEGATIVE_BOARD = 0.25

# Workflow switches. MODE selects full-board or tiled processing.
RUN_BASELINES = False
RUN_REAL_OPTUNA = False
RUN_THRESHOLD_CALIBRATION = False
RUN_LOCKED_TEST = False
RUN_VISUAL_REVIEW = True
FINAL_TRAINING_ENABLED = True
RESUME_FINAL_TRAINING = True
FINAL_RUN_NAME = 'final_detection'
TILED_INFERENCE_CHECKPOINT = None

# Artifact controls.
MODEL_CHECKPOINT = None
BASELINE_CONFIGS = [
    {'name': 'full_or_tiled_baseline', 'mode': MODE, 'model': PRETRAINED_DETECTION_CHECKPOINT, 'imgsz': TRAIN_IMGSZ, 'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0},
]
FORCE_REBUILD = False
FORCE_SPLIT = False
FORCE_TILES = False

for directory in (ARTIFACT_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, LOG_ROOT, OPTUNA_DB_PATH.parent, METRICS_ROOT, REVIEW_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Run name:', RUN_NAME)
print('Mode:', MODE)
print('Configured pretrained checkpoint:', PRETRAINED_DETECTION_CHECKPOINT)
print('Training image size:', TRAIN_IMGSZ)
print('Batch size:', BATCH)
print('Tile geometry:', TILE_WIDTH, 'x', TILE_HEIGHT, 'overlap=', TILE_OVERLAP)
print('Optuna trials:', N_TRIALS)
print('Device:', DEVICE)

## Complete Dataset Audit
This cell reads every manifest entry, label row, and TIFF header. Pixel-level visual analysis remains a separate, deterministic sampled stage because full TIFF decoding can be expensive.

In [ ]:
class_mapping = json.loads(CLASS_MAPPING_PATH.read_text(encoding='utf-8'))
classes = {int(item['id']): item['name'] for item in class_mapping['classes']}
if sorted(classes) != list(range(len(classes))):
    raise ValueError(f'Class IDs must be contiguous from zero: {classes}')

manifest = json.loads(DATASET_MANIFEST_PATH.read_text(encoding='utf-8'))
image_paths = sorted(IMAGE_DIR.glob('*'))
label_paths = sorted(LABEL_DIR.glob('*.txt'))
images_by_stem = {path.stem: path for path in image_paths if path.suffix.lower() in {'.tif', '.tiff'}}
labels_by_stem = {path.stem: path for path in label_paths}
if len(manifest) != len(images_by_stem):
    raise ValueError(f'Manifest/image count mismatch: {len(manifest)} vs {len(images_by_stem)}')

audit_rows = []
box_rows = []
invalid_rows = []
for entry in manifest:
    image_name = Path(entry['image'].replace('\\', '/')).name
    label_name = Path(entry['label'].replace('\\', '/')).name
    stem = Path(image_name).stem
    image_path = IMAGE_DIR / image_name
    label_path = LABEL_DIR / label_name
    if not image_path.exists() or not label_path.exists():
        invalid_rows.append({'stem': stem, 'reason': 'missing image or label'})
        continue
    identity = parse_image_identity(image_path)
    metadata = read_tiff_metadata(image_path)
    lines = label_path.read_text(encoding='utf-8').splitlines()
    nonempty_lines = [line for line in lines if line.strip()]
    if int(entry['box_count']) != len(nonempty_lines):
        invalid_rows.append({'stem': stem, 'reason': 'manifest box_count mismatch'})
    for line_number, line in enumerate(nonempty_lines, start=1):
        fields = line.split()
        if len(fields) != 5:
            invalid_rows.append({'stem': stem, 'line': line_number, 'reason': 'row does not have five fields'})
            continue
        try:
            class_id = int(fields[0])
            values = [finite_or_error(value, f'{stem}:{line_number}') for value in fields[1:]]
        except (TypeError, ValueError) as exc:
            invalid_rows.append({'stem': stem, 'line': line_number, 'reason': str(exc)})
            continue
        cx, cy, box_width, box_height = values
        valid = class_id in classes and 0 <= cx <= 1 and 0 <= cy <= 1 and 0 < box_width <= 1 and 0 < box_height <= 1
        pixel_box = xywhn_to_xyxy(values, metadata['width'], metadata['height'])
        inside = 0 <= pixel_box[0] <= pixel_box[2] <= metadata['width'] and 0 <= pixel_box[1] <= pixel_box[3] <= metadata['height']
        if not valid or not inside:
            invalid_rows.append({'stem': stem, 'line': line_number, 'reason': 'invalid normalized or pixel geometry'})
        box_rows.append({**identity, **metadata, 'class_id': class_id, 'cx': cx, 'cy': cy, 'width': box_width, 'height': box_height, 'x1': pixel_box[0], 'y1': pixel_box[1], 'x2': pixel_box[2], 'y2': pixel_box[3], 'area_fraction': box_width * box_height, 'box_aspect': box_width / box_height})
    audit_rows.append({**identity, **metadata, 'label_path': str(label_path), 'box_count': len(nonempty_lines), 'empty_label': not nonempty_lines})

audit_df = pd.DataFrame(audit_rows)
boxes_df = pd.DataFrame(box_rows)
if invalid_rows:
    invalid_path = METRICS_ROOT / 'invalid_rows.json'
    write_json(invalid_path, invalid_rows)
    raise ValueError(f'{len(invalid_rows)} audit errors. See {invalid_path}')
if set(images_by_stem) != set(labels_by_stem):
    raise ValueError('Image and label stems do not match')

dimension_counts = audit_df.groupby(['width', 'height']).size().to_dict()
summary = {
    'image_count': int(len(audit_df)), 'label_count': int(len(label_paths)), 'box_count': int(len(boxes_df)),
    'empty_label_count': int(audit_df['empty_label'].sum()), 'positive_image_count': int((audit_df['box_count'] > 0).sum()),
    'class_ids_used': sorted(boxes_df['class_id'].unique().tolist()) if not boxes_df.empty else [],
    'dimensions': {f'{width}x{height}': int(count) for (width, height), count in dimension_counts.items()},
    'modes': audit_df['mode'].value_counts().to_dict(), 'formats': audit_df['format'].value_counts().to_dict(),
    'compression': audit_df['compression'].astype(str).value_counts().to_dict(),
    'width': percentile_summary(audit_df['width']), 'height': percentile_summary(audit_df['height']),
    'aspect_ratio': percentile_summary(audit_df['width'] / audit_df['height']),
    'box_width_normalized': percentile_summary(boxes_df['width']) if not boxes_df.empty else {},
    'box_height_normalized': percentile_summary(boxes_df['height']) if not boxes_df.empty else {},
    'box_aspect_ratio': percentile_summary(boxes_df['box_aspect']) if not boxes_df.empty else {},
    'boxes_by_class': boxes_df['class_id'].value_counts().to_dict() if not boxes_df.empty else {},
}
write_json(METRICS_ROOT / 'dataset_audit.json', summary)
audit_df.to_csv(METRICS_ROOT / 'image_audit.csv', index=False)
boxes_df.to_csv(METRICS_ROOT / 'box_audit.csv', index=False)
print(json.dumps(summary, indent=2, default=str))

In [ ]:
# Grouped, persisted split. The leading ID is the conservative leakage key.
split_fingerprint = file_sha256(DATASET_MANIFEST_PATH) + file_sha256(CLASS_MAPPING_PATH)
existing_split = None
if SPLIT_MANIFEST_PATH.exists():
    existing_split = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding='utf-8'))

split_is_current = bool(
    existing_split
    and existing_split.get('dataset_fingerprint') == split_fingerprint
    and existing_split.get('split_algorithm_version') == SPLIT_ALGORITHM_VERSION
    and all(existing_split.get('splits', {}).get(name) for name in ('train', 'val', 'test'))
    and set().union(*(set(existing_split.get('splits', {}).get(name, [])) for name in ('train', 'val', 'test'))) == set(audit_df['image_id'])
)

if split_is_current and not FORCE_SPLIT:
    split_manifest = existing_split
else:
    if existing_split and not FORCE_SPLIT:
        print('Existing split is stale or invalid; regenerating with the corrected split algorithm.')
    rng = np.random.default_rng(SEED)
    groups = audit_df.groupby('group_id', sort=True)
    group_records = []
    for group_id, group in groups:
        group_records.append({'group_id': group_id, 'image_ids': group['image_id'].tolist(), 'size': len(group), 'positive': int(group['box_count'].sum() > 0), 'faces': sorted(group['face'].unique().tolist())})
    rng.shuffle(group_records)
    targets = {'train': len(audit_df) * SPLIT_FRACTIONS['train'], 'val': len(audit_df) * SPLIT_FRACTIONS['val'], 'test': len(audit_df) * SPLIT_FRACTIONS['test']}
    positive_total = sum(record['positive'] for record in group_records)
    positive_targets = {name: positive_total * SPLIT_FRACTIONS[name] for name in targets}
    assigned = {name: [] for name in targets}
    counts = {name: 0 for name in targets}
    positive_counts = {name: 0 for name in targets}
    for record in sorted(group_records, key=lambda item: (item['positive'], item['size']), reverse=True):
        def deficit_score(name):
            image_deficit = max(0.0, targets[name] - counts[name]) / max(targets[name], 1.0)
            positive_deficit = max(0.0, positive_targets[name] - positive_counts[name]) / max(positive_targets[name], 1.0)
            return image_deficit + 0.5 * positive_deficit
        destination = max(targets, key=deficit_score)
        assigned[destination].append(record)
        counts[destination] += record['size']
        positive_counts[destination] += record['positive']
    split_manifest = {'seed': SEED, 'fractions': SPLIT_FRACTIONS, 'group_key': GROUP_KEY, 'split_algorithm_version': SPLIT_ALGORITHM_VERSION, 'dataset_fingerprint': split_fingerprint, 'splits': {name: [image_id for group in records for image_id in group['image_ids']] for name, records in assigned.items()}, 'groups': {name: [group['group_id'] for group in records] for name, records in assigned.items()}, 'group_positive_counts': positive_counts}
    write_json(SPLIT_MANIFEST_PATH, split_manifest)

split_sets = {name: set(values) for name, values in split_manifest['splits'].items()}
if split_sets['train'] & split_sets['val'] or split_sets['train'] & split_sets['test'] or split_sets['val'] & split_sets['test']:
    raise AssertionError('Image IDs overlap between splits')
group_sets = {name: set(values) for name, values in split_manifest['groups'].items()}
if group_sets['train'] & group_sets['val'] or group_sets['train'] & group_sets['test'] or group_sets['val'] & group_sets['test']:
    raise AssertionError('Group IDs overlap between splits')
if not split_sets['train'] or not split_sets['val'] or not split_sets['test']:
    raise AssertionError(f'Every split must contain images; got { {name: len(values) for name, values in split_sets.items()} }')
if set().union(*split_sets.values()) != set(audit_df['image_id']):
    raise AssertionError('Split manifest does not cover every audited image exactly once')
print('Image counts:', {name: len(values) for name, values in split_manifest['splits'].items()})
print('Group counts:', {name: len(values) for name, values in split_manifest['groups'].items()})
print('Positive group counts:', split_manifest.get('group_positive_counts', 'legacy split'))

In [ ]:
# Create an idempotent Ultralytics detection dataset. Empty labels remain empty negatives.
def prepare_yolo_dataset(source_images=IMAGE_DIR, source_labels=LABEL_DIR, output_root=YOLO_DATASET_ROOT):
    if (output_root / 'data.yaml').exists() and not FORCE_REBUILD:
        return output_root / 'data.yaml'
    for split in ('train', 'val', 'test'):
        (output_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_root / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for image_id in split_manifest['splits'][split]:
            image_path = images_by_stem[image_id]
            label_path = source_labels / f'{image_id}.txt'
            destination_image = output_root / 'images' / split / image_path.name
            destination_label = output_root / 'labels' / split / label_path.name
            if destination_image.exists() and destination_label.exists() and not FORCE_REBUILD:
                continue
            shutil.copy2(image_path, destination_image)
            shutil.copy2(label_path, destination_label)
    payload = {'path': str(output_root.resolve()), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'Pith'}, 'nc': 1}
    (output_root / 'data.yaml').write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
    return output_root / 'data.yaml'

data_yaml_path = prepare_yolo_dataset()
print(data_yaml_path)

In [ ]:
# Geometry tests and pre-training transform resolution report.
assert xywhn_to_xyxy((0.5, 0.5, 0.4, 0.2), 100, 50) == (30.0, 20.0, 70.0, 30.0)
assert xyxy_to_xywhn((30, 20, 70, 30), 100, 50) == (0.5, 0.5, 0.4, 0.2)
assert box_iou((0, 0, 10, 10), (5, 5, 15, 15)) == 25 / 175
if not boxes_df.empty:
    height = int(audit_df.iloc[0]['height'])
    width = int(audit_df.iloc[0]['width'])
    scale = min(TRAIN_IMGSZ / width, TRAIN_IMGSZ / height)
    boxes_df['resized_width_pixels'] = boxes_df['width'] * width * scale
    boxes_df['resized_height_pixels'] = boxes_df['height'] * height * scale
    print('Candidate full-board retained box pixels:', boxes_df[['resized_width_pixels', 'resized_height_pixels']].describe(percentiles=[.1, .5, .9]).to_dict())
print('Geometry tests passed')

## Smoke Test and Native Training
The following cells deliberately use the installed Ultralytics interface. They do not export models. Run the smoke test before any expensive training.

In [ ]:
# Runtime support checks and model selection.
MODEL_CHECKPOINT = PRETRAINED_DETECTION_CHECKPOINT
checkpoint_path = Path(MODEL_CHECKPOINT)
if not checkpoint_path.exists() and not MODEL_CHECKPOINT.startswith(('yolo', 'YOLO')):
    raise FileNotFoundError(f'Configured pretrained checkpoint does not exist: {checkpoint_path}')
if 'seg' in checkpoint_path.name.lower() or 'seg' in MODEL_CHECKPOINT.lower():
    raise ValueError(f'Configured checkpoint appears to be a segmentation model: {MODEL_CHECKPOINT}')
print('Selected smoke-test checkpoint:', MODEL_CHECKPOINT)
smoke_model = YOLO(MODEL_CHECKPOINT)
if getattr(smoke_model, 'task', None) not in (None, 'detect'):
    raise RuntimeError(f'Expected a detection checkpoint, got task={smoke_model.task!r}. Select a native detect .pt model.')
print('Model task:', getattr(smoke_model, 'task', None), 'names:', getattr(smoke_model, 'names', None))

TRAIN_KWARG_NAMES = set(inspect.signature(smoke_model.train).parameters)
PREDICT_KWARG_NAMES = set(inspect.signature(smoke_model.predict).parameters)
VAL_KWARG_NAMES = set(inspect.signature(smoke_model.val).parameters)
print('Validated train kwargs:', len(TRAIN_KWARG_NAMES), 'predict kwargs:', len(PREDICT_KWARG_NAMES), 'val kwargs:', len(VAL_KWARG_NAMES))

In [ ]:
# Mandatory short smoke test.
smoke_run = MODEL_ROOT / 'smoke_test'
smoke_train_kwargs = {
    'data': str(data_yaml_path), 'epochs': 1, 'imgsz': TRAIN_IMGSZ, 'batch': 1,
    'device': DEVICE, 'rect': RECT, 'workers': 0, 'cache': False, 'amp': False,
    'patience': 1, 'save': True, 'project': str(MODEL_ROOT), 'name': 'smoke_test',
    'exist_ok': True, 'verbose': False, 'mosaic': 0.0, 'mixup': 0.0,
    'copy_paste': 0.0, 'deterministic': DETERMINISTIC,
}
if not (smoke_run / 'weights' / 'last.pt').exists():
    smoke_model.train(**supported_kwargs(smoke_model.train, smoke_train_kwargs))
else:
    print('Smoke test checkpoint exists; refusing to overwrite it.')
assert (smoke_run / 'weights' / 'last.pt').exists(), 'Smoke test did not produce last.pt'
print('Smoke test passed:', smoke_run / 'weights' / 'last.pt')

In [ ]:
# Controlled baseline runner. Set RUN_BASELINES=True in the central configuration cell.
def print_gpu_memory(label=''):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / 2**30
        reserved = torch.cuda.memory_reserved() / 2**30
        total = torch.cuda.get_device_properties(DEVICE).total_memory / 2**30
        print(f'[GPU memory {label}] allocated={allocated:.2f} GiB reserved={reserved:.2f} GiB total={total:.2f} GiB')


def train_with_oom_backoff(model_factory, train_kwargs, min_batch=1):
    kwargs = dict(train_kwargs)
    model = model_factory()
    while True:
        try:
            print_gpu_memory('before train')
            model.train(**supported_kwargs(model.train, kwargs))
            print_gpu_memory('after train')
            return model
        except (RuntimeError, torch.cuda.OutOfMemoryError) as exc:
            if 'out of memory' not in str(exc).lower():
                raise

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, 'ipc_collect'):
                    torch.cuda.ipc_collect()

            # In notebook + Linux environments, DataLoader worker forking is a common instability source.
            changes = []
            if kwargs.get('workers', 0) > 0:
                kwargs['workers'] = 0
                changes.append('workers=0')
            elif kwargs.get('amp', False):
                kwargs['amp'] = False
                changes.append('amp=False')
            elif kwargs['batch'] > min_batch:
                kwargs['batch'] = max(min_batch, kwargs['batch'] // 2)
                changes.append(f'batch={kwargs["batch"]}')
            else:
                raise RuntimeError(
                    f'CUDA OOM persisted at workers={kwargs.get("workers")}, amp={kwargs.get("amp")}, batch={kwargs.get("batch")}. '
                    'Try lowering TRAIN_IMGSZ or reducing model size.'
                ) from exc

            print(f'CUDA OOM caught; rebuilding model and retrying with {", ".join(changes)}')
            model = model_factory()


baseline_results = {}
if RUN_BASELINES:
    for config in BASELINE_CONFIGS:
        run_dir = MODEL_ROOT / config['name']
        best_path = run_dir / 'weights' / 'best.pt'
        if best_path.exists():
            print('Existing baseline:', best_path)
            continue
        train_kwargs = {'data': str(data_yaml_path), 'epochs': EPOCHS, 'imgsz': config['imgsz'], 'batch': BATCH, 'device': DEVICE, 'rect': RECT, 'workers': NUM_WORKERS, 'cache': CACHE, 'amp': AMP, 'patience': PATIENCE, 'save': True, 'save_period': SAVE_PERIOD, 'project': str(MODEL_ROOT), 'name': config['name'], 'exist_ok': False, 'mosaic': config['mosaic'], 'mixup': config['mixup'], 'copy_paste': config['copy_paste'], 'verbose': True, 'deterministic': DETERMINISTIC}
        model = train_with_oom_backoff(lambda: YOLO(config['model']), train_kwargs)
        baseline_results[config['name']] = str(best_path)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print('Baselines disabled. Set RUN_BASELINES=True in the central configuration cell.')
print(baseline_results)

## Optuna, Calibration, Locked Test, and Native `.pt` Verification
These stages are intentionally guarded. The objective must use validation only; the test split is not read until the locked-test cell.

In [ ]:
# Persistent Optuna study. Set RUN_REAL_OPTUNA=True in the central configuration cell to run trials.
if RUN_REAL_OPTUNA:
    import optuna
    storage = f'sqlite:///{OPTUNA_DB_PATH.resolve()}'
    sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
    pruner = optuna.pruners.HyperbandPruner(min_resource=5, max_resource=TRIAL_EPOCHS, reduction_factor=3)
    study = optuna.create_study(direction='maximize', study_name=OPTUNA_STUDY_NAME, storage=storage, load_if_exists=True, sampler=sampler, pruner=pruner)
    print('Persistent study:', study.study_name, 'trials:', len(study.trials))
else:
    print('Optuna disabled. Set RUN_REAL_OPTUNA=True in the central configuration cell.')

In [ ]:
# Native checkpoint metadata and deployment configuration helper.
def save_deployment_config(checkpoint_path, confidence, nms_iou, mode='full'):
    payload = {
        'checkpoint_path': str(Path(checkpoint_path).resolve()),
        'class_mapping': classes,
        'image_format': ['.tif', '.tiff'],
        'tiff_loading': 'Pillow, convert RGB',
        'input_resolution': TRAIN_IMGSZ,
        'rectangular_preprocessing': RECT,
        'confidence_threshold': float(confidence),
        'nms_iou_threshold': float(nms_iou),
        'mode': mode,
        'tile_width': TILE_WIDTH, 'tile_height': TILE_HEIGHT, 'tile_overlap': TILE_OVERLAP,
        'tile_box_retention': MIN_RETAINED_BOX_FRACTION,
        'merge_method': MERGE_METHOD, 'merge_iou': MERGE_IOU,
        'device': str(DEVICE), 'amp': AMP, 'ultralytics_version': ULTRALYTICS_VERSION, 'pytorch_version': torch.__version__,
    }
    write_json(DEPLOYMENT_CONFIG_PATH, payload)
    return payload

def native_pt_inference(checkpoint_path, image_path, confidence=0.25, nms_iou=0.7, save_path=None):
    model = YOLO(str(checkpoint_path))
    results = model.predict(source=str(image_path), conf=confidence, iou=nms_iou, imgsz=TRAIN_IMGSZ, rect=RECT, device=DEVICE, save=False, verbose=False)
    result = results[0]
    detections = []
    if result.boxes is not None:
        for xyxy, conf, cls in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy()):
            detections.append({'xyxy': [float(value) for value in xyxy], 'confidence': float(conf), 'class_id': int(cls), 'class_name': classes.get(int(cls), str(int(cls)))})
    if save_path is not None:
        result.save(filename=str(save_path))
    del model
    return detections

print('Native PyTorch inference helper defined; no export path exists in this notebook.')

In [ ]:
# Reload verification. Set BEST_CHECKPOINT after final training and calibration.
BEST_CHECKPOINT = None
SELECTED_CONFIDENCE = None
SELECTED_NMS_IOU = None
if BEST_CHECKPOINT and SELECTED_CONFIDENCE is not None and SELECTED_NMS_IOU is not None:
    sample_paths = [images_by_stem[image_id] for image_id in split_manifest['splits']['val'][:3]]
    references = [native_pt_inference(BEST_CHECKPOINT, path, SELECTED_CONFIDENCE, SELECTED_NMS_IOU) for path in sample_paths]
    del references
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    reloaded = YOLO(str(BEST_CHECKPOINT))
    checkpoint_names = {int(key): value for key, value in reloaded.names.items()} if isinstance(reloaded.names, dict) else reloaded.names
    if checkpoint_names != {0: 'Pith'}:
        raise AssertionError(f'Unexpected checkpoint class mapping: {checkpoint_names}')
    print('Reloaded native .pt checkpoint:', BEST_CHECKPOINT)
    save_deployment_config(BEST_CHECKPOINT, SELECTED_CONFIDENCE, SELECTED_NMS_IOU, MODE)
else:
    print('Reload verification is locked until BEST_CHECKPOINT and validation thresholds are supplied.')

In [ ]:
# Mode-specific training data, validation calibration, Optuna, and locked test evaluation.
from functions.yolo_detection import binary_detection_metrics, match_detections, read_yolo_boxes, tile_label_rows


def prepare_tiled_detection_dataset():
    if (TILE_DATASET_ROOT / 'tile_manifest.json').exists() and not FORCE_TILES:
        return TILE_DATASET_ROOT
    for split in ('train', 'val', 'test'):
        (TILE_DATASET_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
        (TILE_DATASET_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
    manifest = {}
    positive_count = defaultdict(int)
    negative_candidates = []
    rng = np.random.default_rng(SEED)
    for split in ('train', 'val', 'test'):
        for image_id in split_manifest['splits'][split]:
            image_path = images_by_stem[image_id]
            label_path = labels_by_stem[image_id]
            with Image.open(image_path) as source:
                image = source.convert('RGB')
                width, height = image.size
                identity = parse_image_identity(image_path)
                for index, tile in enumerate(tile_windows(height, width, TILE_HEIGHT, TILE_WIDTH, TILE_OVERLAP)):
                    rows = tile_label_rows(label_path, width, height, tile, MIN_RETAINED_BOX_FRACTION, MIN_RETAINED_BOX_WIDTH, MIN_RETAINED_BOX_HEIGHT)
                    record = {'tile_id': f'{image_id}__tile_{index:04d}', 'parent_image_id': image_id, 'group_id': identity['group_id'], 'split': split, 'tile': list(tile), 'parent_size': [height, width], 'box_count': len(rows)}
                    if split == 'train' and not rows:
                        negative_candidates.append((record, image.copy()))
                        continue
                    y1, y2, x1, x2 = tile
                    image.crop((x1, y1, x2, y2)).save(TILE_DATASET_ROOT / 'images' / split / f"{record['tile_id']}.tif")
                    (TILE_DATASET_ROOT / 'labels' / split / f"{record['tile_id']}.txt").write_text(''.join(' '.join([str(row[0])] + [f'{value:.8f}' for value in row[1:]]) + '\n' for row in rows), encoding='utf-8')
                    manifest[record['tile_id']] = record
                    positive_count[split] += int(bool(rows))
    rng.shuffle(negative_candidates)
    keep = min(len(negative_candidates), int(positive_count['train'] * NEGATIVE_TILE_RATIO))
    for record, image in negative_candidates[:keep]:
        y1, y2, x1, x2 = record['tile']
        image.crop((x1, y1, x2, y2)).save(TILE_DATASET_ROOT / 'images' / 'train' / f"{record['tile_id']}.tif")
        (TILE_DATASET_ROOT / 'labels' / 'train' / f"{record['tile_id']}.txt").write_text('', encoding='utf-8')
        manifest[record['tile_id']] = record
    write_json(TILE_DATASET_ROOT / 'tile_manifest.json', manifest)
    tile_yaml = {'path': str(TILE_DATASET_ROOT.resolve()), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'Pith'}, 'nc': 1}
    (TILE_DATASET_ROOT / 'data.yaml').write_text(yaml.safe_dump(tile_yaml, sort_keys=False), encoding='utf-8')
    write_json(TILE_DATASET_ROOT / 'tile_stats.json', {'tiles': len(manifest), 'positive_tiles': positive_count, 'negative_train_tiles_kept': keep})
    return TILE_DATASET_ROOT


def _prediction_rows(result, confidence):
    if result.boxes is None:
        return []
    return [(*[float(value) for value in xyxy], float(score), int(class_id)) for xyxy, score, class_id in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy()) if float(score) >= confidence]


def _board_truth(image_path):
    with Image.open(image_path) as source:
        width, height = source.size
    label_path = labels_by_stem[Path(image_path).stem]
    truth = [xywhn_to_xyxy(row[1:], width, height) for row in read_yolo_boxes(label_path)]
    return truth, width, height


def evaluate_detection_model(model, image_paths, confidence=0.25, nms_iou=0.7):
    total = {'tp': 0, 'fp': 0, 'fn': 0}
    negative_boards = 0
    negative_boards_with_fp = 0
    records = []
    for image_path in image_paths:
        truth, _, _ = _board_truth(image_path)
        result = model.predict(source=str(image_path), conf=0.001, iou=nms_iou, imgsz=TRAIN_IMGSZ, rect=RECT, device=DEVICE, save=False, verbose=False)[0]
        predictions = _prediction_rows(result, confidence)
        match = match_detections([row[:5] for row in predictions], truth)
        for key in total:
            total[key] += match[key]
        if not truth:
            negative_boards += 1
            negative_boards_with_fp += int(match['fp'] > 0)
        records.append({'image_id': Path(image_path).stem, **match, 'predictions': predictions, 'truth': truth})
    metrics = binary_detection_metrics(total)
    positive_records = [row for row in records if row['tp'] + row['fn'] > 0]
    metrics.update({'tp': total['tp'], 'fp': total['fp'], 'fn': total['fn'], 'negative_board_count': negative_boards, 'negative_boards_with_fp': negative_boards_with_fp, 'fp_per_negative_board': total['fp'] / max(negative_boards, 1), 'negative_board_fp_rate': negative_boards_with_fp / max(negative_boards, 1), 'board_sensitivity': sum(row['tp'] > 0 for row in positive_records) / max(len(positive_records), 1), 'inference_mode': 'full'})
    return metrics, records


def evaluate_tiled_detection_model(model, image_paths, confidence=0.25, nms_iou=0.7):
    from torchvision.ops import nms

    total = {'tp': 0, 'fp': 0, 'fn': 0}
    negative_boards = 0
    negative_boards_with_fp = 0
    records = []
    for image_path in image_paths:
        truth, width, height = _board_truth(image_path)
        with Image.open(image_path) as source:
            image = source.convert('RGB')
            tile_predictions = []
            for tile in tile_windows(height, width, TILE_HEIGHT, TILE_WIDTH, TILE_OVERLAP):
                y1, y2, x1, x2 = tile
                tile_image = image.crop((x1, y1, x2, y2))
                tile_result = model.predict(source=tile_image, conf=0.001, iou=nms_iou, imgsz=TRAIN_IMGSZ, rect=RECT, device=DEVICE, save=False, verbose=False)[0]
                for x1, y1, x2, y2, score, class_id in _prediction_rows(tile_result, 0.001):
                    restored = restore_tile_box((x1, y1, x2, y2), tile)
                    tile_predictions.append((*restored, score, class_id))
        if tile_predictions:
            boxes = torch.tensor([row[:4] for row in tile_predictions], dtype=torch.float32, device=DEVICE)
            scores = torch.tensor([row[4] for row in tile_predictions], dtype=torch.float32, device=DEVICE)
            keep = nms(boxes, scores, nms_iou).cpu().tolist()
            predictions = [tile_predictions[index] for index in keep if tile_predictions[index][4] >= confidence]
        else:
            predictions = []
        match = match_detections([row[:5] for row in predictions], truth)
        for key in total:
            total[key] += match[key]
        if not truth:
            negative_boards += 1
            negative_boards_with_fp += int(match['fp'] > 0)
        records.append({'image_id': Path(image_path).stem, **match, 'predictions': predictions, 'truth': truth})
    metrics = binary_detection_metrics(total)
    positive_records = [row for row in records if row['tp'] + row['fn'] > 0]
    metrics.update({'tp': total['tp'], 'fp': total['fp'], 'fn': total['fn'], 'negative_board_count': negative_boards, 'negative_boards_with_fp': negative_boards_with_fp, 'fp_per_negative_board': total['fp'] / max(negative_boards, 1), 'negative_board_fp_rate': negative_boards_with_fp / max(negative_boards, 1), 'board_sensitivity': sum(row['tp'] > 0 for row in positive_records) / max(len(positive_records), 1), 'inference_mode': 'tiled', 'tile_width': TILE_WIDTH, 'tile_height': TILE_HEIGHT, 'tile_overlap': TILE_OVERLAP, 'merge_iou': nms_iou})
    return metrics, records


def evaluate_mode(model, image_paths, confidence=0.25, nms_iou=0.7):
    evaluator = evaluate_tiled_detection_model if MODE == 'tiled' else evaluate_detection_model
    return evaluator(model, image_paths, confidence, nms_iou)


def validation_operating_point(model):
    validation_paths = [images_by_stem[image_id] for image_id in split_manifest['splits']['val']]
    candidates = []
    for nms_iou in NMS_IOU_GRID:
        for confidence in CONFIDENCE_GRID:
            metrics, _ = evaluate_mode(model, validation_paths, confidence, nms_iou)
            eligible = (
                metrics['recall'] >= MIN_RECALL
                and metrics['fp_per_negative_board'] <= MAX_FP_PER_NEGATIVE_BOARD
            )
            candidates.append({
                'confidence': float(confidence),
                'nms_iou': float(nms_iou),
                'eligible': bool(eligible),
                'selection_score': metrics['recall'] + metrics['precision'] + metrics['f1'],
                **metrics,
            })

    strict = [row for row in candidates if row['eligible']]

    # Relaxed fallback window: small recall drop allowed, but still FP-focused.
    relaxed = [
        row for row in candidates
        if row['recall'] >= max(0.0, MIN_RECALL - 0.15)
        and row['fp_per_negative_board'] <= (MAX_FP_PER_NEGATIVE_BOARD * 2.0)
    ]

    if strict:
        selected = max(
            strict,
            key=lambda row: (
                row['f1'],
                row['recall'],
                row['precision'],
                -row['fp_per_negative_board'],
                -row['negative_board_fp_rate'],
            ),
        )
        selected['selection_reason'] = 'strict_constraints_met'
    elif relaxed:
        selected = min(
            relaxed,
            key=lambda row: (
                row['fp_per_negative_board'],
                row['negative_board_fp_rate'],
                -row['recall'],
                -row['f1'],
                -row['confidence'],   # prefer higher confidence when tied
                row['nms_iou'],       # prefer lower NMS IoU when tied
            ),
        )
        selected['selection_reason'] = 'relaxed_fp_first'
    else:
        selected = min(
            candidates,
            key=lambda row: (
                row['fp_per_negative_board'],
                row['negative_board_fp_rate'],
                -row['confidence'],
                row['nms_iou'],
                -row['recall'],
            ),
        )
        selected['selection_reason'] = 'safety_min_fp'

    write_json(METRICS_ROOT / 'validation_threshold_grid.json', candidates)
    write_json(METRICS_ROOT / 'validation_operating_point.json', selected)
    return selected


if MODE == 'tiled':
    tiled_root = prepare_tiled_detection_dataset()
    print('Tiled dataset ready:', tiled_root)
if RUN_REAL_OPTUNA:
    import optuna
    storage = f"sqlite:///{OPTUNA_DB_PATH.resolve()}"
    study = optuna.create_study(direction='maximize', study_name=OPTUNA_STUDY_NAME, storage=storage, load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True), pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=TRIAL_EPOCHS, reduction_factor=3))
    def objective(trial):
        trial_imgsz = trial.suggest_categorical('imgsz', FULL_BOARD_IMGSZ_CANDIDATES)
        trial_config = {'lr0': trial.suggest_float('lr0', 1e-4, 3e-2, log=True), 'lrf': trial.suggest_float('lrf', 0.01, 0.3, log=True), 'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True), 'optimizer': trial.suggest_categorical('optimizer', ['SGD', 'AdamW'])}
        trial_name = f'trial_{trial.number:04d}'
        train_kwargs = {
            'data': str(TILE_DATASET_ROOT / 'data.yaml' if MODE == 'tiled' else data_yaml_path),
            'epochs': TRIAL_EPOCHS, 'imgsz': trial_imgsz, 'batch': BATCH, 'device': DEVICE,
            'rect': RECT, 'workers': 0, 'cache': CACHE, 'amp': False, 'patience': 8,
            'project': str(MODEL_ROOT / 'optuna'), 'name': trial_name, 'exist_ok': False,
            'save': True, 'verbose': False,
            'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0, 'deterministic': DETERMINISTIC,
            **trial_config}
        def trial_model_factory():
            # Report Ultralytics' per-epoch validation fitness so the Hyperband pruner can
            # stop weak trials early. The final objective below is still the composite score.
            trial_model = YOLO(MODEL_CHECKPOINT)

            def report_epoch(trainer):
                if trainer.fitness is None:
                    return
                epoch = trainer.epoch + 1
                trial.report(float(trainer.fitness), epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned(f'Pruned at epoch {epoch}')

            trial_model.add_callback('on_fit_epoch_end', report_epoch)
            return trial_model

        model = train_with_oom_backoff(trial_model_factory, train_kwargs)
        validation_data = TILE_DATASET_ROOT / 'data.yaml' if MODE == 'tiled' else data_yaml_path
        val_results = model.val(data=str(validation_data), split='val', imgsz=trial_imgsz, batch=BATCH, device=DEVICE, rect=RECT, plots=False, verbose=False)
        box_metrics = getattr(val_results, 'box', None)
        ap50 = float(getattr(box_metrics, 'map50', 0.0))
        ap50_95 = float(getattr(box_metrics, 'map', 0.0))
        custom_metrics, _ = evaluate_mode(model, [images_by_stem[image_id] for image_id in split_manifest['splits']['val']], 0.25, 0.7)
        objective_value = 0.30 * custom_metrics['recall'] + 0.20 * ap50 + 0.15 * ap50_95 + 0.15 * custom_metrics['precision'] + 0.10 * custom_metrics['board_sensitivity'] + 0.10 * custom_metrics['f1'] - 0.10 * min(1.0, custom_metrics['fp_per_negative_board'] / max(MAX_FP_PER_NEGATIVE_BOARD, 1e-6))
        trial.set_user_attr('ap50', ap50)
        trial.set_user_attr('ap50_95', ap50_95)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return objective_value
    study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, catch=(RuntimeError, torch.cuda.OutOfMemoryError))
    write_json(METRICS_ROOT / 'optuna_best.json', {'value': study.best_value, 'params': study.best_params})
if RUN_THRESHOLD_CALIBRATION:
    final_model = YOLO(str(BEST_CHECKPOINT))
    selected_thresholds = validation_operating_point(final_model)
    SELECTED_CONFIDENCE = selected_thresholds['confidence']
    SELECTED_NMS_IOU = selected_thresholds['nms_iou']
    del final_model
if RUN_LOCKED_TEST:
    if SELECTED_CONFIDENCE is None or SELECTED_NMS_IOU is None:
        raise RuntimeError('Freeze validation thresholds before locked test evaluation.')
    test_model = YOLO(str(BEST_CHECKPOINT))
    test_paths = [images_by_stem[image_id] for image_id in split_manifest['splits']['test']]
    locked_metrics, locked_records = evaluate_mode(test_model, test_paths, SELECTED_CONFIDENCE, SELECTED_NMS_IOU)
    write_json(METRICS_ROOT / 'locked_test_metrics.json', locked_metrics)
    pd.DataFrame(locked_records).to_json(METRICS_ROOT / 'locked_test_records.json', orient='records', indent=2)
    print(json.dumps(locked_metrics, indent=2))
    del test_model

In [ ]:
# Final locked training with native PyTorch checkpoints.
if FINAL_TRAINING_ENABLED:
    FINAL_RUN_DIR = MODEL_ROOT / FINAL_RUN_NAME
    FINAL_BEST_CHECKPOINT = FINAL_RUN_DIR / 'weights' / 'best.pt'
    FINAL_LAST_CHECKPOINT = FINAL_RUN_DIR / 'weights' / 'last.pt'
    FINAL_DATA_YAML = TILE_DATASET_ROOT / 'data.yaml' if MODE == 'tiled' else data_yaml_path
    if MODE == 'tiled' and not FINAL_DATA_YAML.exists():
        raise FileNotFoundError(f'Tiled dataset YAML not found: {FINAL_DATA_YAML}. Run the dataset preparation cell first.')

    prior_config_path = MODEL_ROOT / 'final_training_config.json'
    prior_mode = None
    if prior_config_path.exists():
        prior_mode = json.loads(prior_config_path.read_text(encoding='utf-8')).get('mode')
    resume_allowed = RESUME_FINAL_TRAINING and prior_mode in (None, MODE)
    resume_from_last = FINAL_LAST_CHECKPOINT.exists() and resume_allowed
    if FINAL_LAST_CHECKPOINT.exists() and RESUME_FINAL_TRAINING and prior_mode not in (None, MODE):
        print(f'Refusing to resume {prior_mode!r} checkpoint in MODE={MODE!r}; starting from {MODEL_CHECKPOINT}.')

    selected_params = {}
    if 'study' in globals() and getattr(study, 'best_trial', None) is not None:
        selected_params = dict(study.best_trial.params)
    selected_params.pop('model', None)
    selected_params.pop('imgsz', None)
    selected_params.pop('epochs', None)
    selected_params.pop('batch', None)
    selected_params.pop('device', None)
    selected_params.pop('data', None)
    final_train_kwargs = {
        'data': str(FINAL_DATA_YAML), 'epochs': EPOCHS, 'imgsz': TRAIN_IMGSZ,
        'batch': BATCH, 'device': DEVICE, 'rect': RECT, 'workers': NUM_WORKERS,
        'cache': CACHE, 'amp': AMP, 'patience': PATIENCE, 'save': True,
        'save_period': SAVE_PERIOD, 'project': str(MODEL_ROOT),
        'name': FINAL_RUN_NAME, 'exist_ok': True, 'verbose': True,
        'resume': resume_from_last, 'mosaic': 0.0, 'mixup': 0.0,
        'copy_paste': 0.0, 'deterministic': DETERMINISTIC, **selected_params,
    }

    def make_final_model():
        if final_train_kwargs.get('resume', False):
            return YOLO(str(FINAL_LAST_CHECKPOINT))
        return YOLO(MODEL_CHECKPOINT)

    final_model = train_with_oom_backoff(make_final_model, final_train_kwargs)
    if not FINAL_BEST_CHECKPOINT.exists() or not FINAL_LAST_CHECKPOINT.exists():
        raise RuntimeError(f'Final training did not produce both native checkpoints: {FINAL_BEST_CHECKPOINT}, {FINAL_LAST_CHECKPOINT}')

    final_config = {
        'mode': MODE,
        'pretrained_checkpoint': str(Path(MODEL_CHECKPOINT).resolve()),
        'final_run_dir': str(FINAL_RUN_DIR.resolve()),
        'best_checkpoint': str(FINAL_BEST_CHECKPOINT.resolve()),
        'last_checkpoint': str(FINAL_LAST_CHECKPOINT.resolve()),
        'resume_requested': RESUME_FINAL_TRAINING,
        'resumed_from_last': resume_from_last,
        'training_args': final_train_kwargs,
        'class_mapping': classes,
        'dataset_yaml': str(FINAL_DATA_YAML.resolve()),
        'split_manifest': str(SPLIT_MANIFEST_PATH.resolve()),
        'ultralytics_version': ULTRALYTICS_VERSION,
        'pytorch_version': torch.__version__,
    }
    write_json(MODEL_ROOT / 'final_training_config.json', final_config)
    print('Final mode:', MODE)
    print('Final dataset:', FINAL_DATA_YAML)
    print('Final best checkpoint:', FINAL_BEST_CHECKPOINT)
    print('Final last checkpoint:', FINAL_LAST_CHECKPOINT)
    del final_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('Final training disabled. Set FINAL_TRAINING_ENABLED=True in the central configuration cell.')

In [ ]:
# Final validation calibration and one-time locked-test evaluation.
# This cell must run after final training and uses the final native best.pt.
if not FINAL_TRAINING_ENABLED:
    print('Skipping final evaluation because FINAL_TRAINING_ENABLED=False.')
else:
    FINAL_BEST_CHECKPOINT = MODEL_ROOT / 'final_detection' / 'weights' / 'best.pt'
    if not FINAL_BEST_CHECKPOINT.exists():
        raise FileNotFoundError(f'Final best checkpoint not found: {FINAL_BEST_CHECKPOINT}')

    final_eval_model = YOLO(str(FINAL_BEST_CHECKPOINT))
    BEST_CHECKPOINT = str(FINAL_BEST_CHECKPOINT)
    validation_thresholds = validation_operating_point(final_eval_model)
    SELECTED_CONFIDENCE = validation_thresholds['confidence']
    SELECTED_NMS_IOU = validation_thresholds['nms_iou']
    write_json(METRICS_ROOT / 'final_validation_operating_point.json', validation_thresholds)

    test_paths = [images_by_stem[image_id] for image_id in split_manifest['splits']['test']]
    test_metrics, test_records = evaluate_mode(final_eval_model, test_paths, SELECTED_CONFIDENCE, SELECTED_NMS_IOU)

    evaluation_data_yaml = TILE_DATASET_ROOT / 'data.yaml' if MODE == 'tiled' else data_yaml_path
    val_results = final_eval_model.val(
        data=str(evaluation_data_yaml),
        split='test',
        imgsz=TRAIN_IMGSZ,
        batch=BATCH,
        device=DEVICE,
        rect=RECT,
        plots=True,
        project=str(METRICS_ROOT),
        name='ultralytics_test_validation',
        exist_ok=True,
        verbose=True,
    )
    box_metrics = getattr(val_results, 'box', None)
    test_metrics['tile_dataset_ap50'] = float(getattr(box_metrics, 'map50', 0.0)) if MODE == 'tiled' else None
    test_metrics['tile_dataset_ap50_95'] = float(getattr(box_metrics, 'map', 0.0)) if MODE == 'tiled' else None

    test_metrics['selected_confidence'] = float(SELECTED_CONFIDENCE)
    test_metrics['selected_nms_iou'] = float(SELECTED_NMS_IOU)
    test_metrics['evaluation_split'] = 'test'
    test_metrics['test_board_count'] = len(test_paths)
    write_json(METRICS_ROOT / 'locked_test_metrics.json', test_metrics)
    pd.DataFrame(test_records).to_json(METRICS_ROOT / 'locked_test_records.json', orient='records', indent=2)
    save_deployment_config(BEST_CHECKPOINT, SELECTED_CONFIDENCE, SELECTED_NMS_IOU, MODE)
    print(json.dumps(test_metrics, indent=2, default=str))
    del final_eval_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Visual metrics and qualitative review for the locked test set.
if not RUN_VISUAL_REVIEW:
    print('Visual review disabled. Set RUN_VISUAL_REVIEW=True in the central configuration cell.')
else:
    import matplotlib.pyplot as plt
    from PIL import ImageDraw
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, roc_curve

    visual_dir = METRICS_ROOT / 'visual_review'
    visual_dir.mkdir(parents=True, exist_ok=True)
    visual_model = YOLO(str(FINAL_BEST_CHECKPOINT))
    # Predict the same way as the locked test (tiled + merged in tiled MODE, whole board otherwise),
    # keeping every box down to conf 0.001 so the confidence sweep below can threshold them.
    visual_paths = [images_by_stem[image_id] for image_id in split_manifest['splits']['test']]
    _, visual_records = evaluate_mode(visual_model, visual_paths, confidence=0.001, nms_iou=SELECTED_NMS_IOU)
    visual_rows = [
        {'image_id': record['image_id'], 'image_path': str(image_path), 'truth': record['truth'], 'predictions': record['predictions'], 'image_positive': bool(record['truth'])}
        for record, image_path in zip(visual_records, visual_paths)
    ]

    sweep_rows = []
    for confidence in CONFIDENCE_GRID:
        aggregate = {'tp': 0, 'fp': 0, 'fn': 0}
        scores, targets = [], []
        for row in visual_rows:
            predictions = [item for item in row['predictions'] if item[4] >= confidence]
            match = match_detections([item[:5] for item in predictions], row['truth'])
            for key in aggregate:
                aggregate[key] += match[key]
            scores.append(max((item[4] for item in row['predictions']), default=0.0))
            targets.append(int(row['image_positive']))
        sweep_rows.append({'confidence': float(confidence), **binary_detection_metrics(aggregate), **aggregate, 'scores': scores, 'targets': targets})

    sweep_df = pd.DataFrame([{key: value for key, value in row.items() if key not in ('scores', 'targets')} for row in sweep_rows])
    sweep_df.to_csv(visual_dir / 'confidence_sweep_metrics.csv', index=False)
    scores, targets = sweep_rows[0]['scores'], sweep_rows[0]['targets']
    predicted = [int(score >= SELECTED_CONFIDENCE) for score in scores]
    roc_auc = float(roc_auc_score(targets, scores)) if len(set(targets)) == 2 else None

    figure, axes = plt.subplots(2, 2, figsize=(12, 9))
    for metric in ('precision', 'recall', 'f1'):
        axes[0, 0].plot(sweep_df['confidence'], sweep_df[metric], label=metric)
    axes[0, 0].axvline(SELECTED_CONFIDENCE, color='black', linestyle='--')
    axes[0, 0].set_title('Box metrics versus confidence')
    axes[0, 0].legend()
    for metric in ('tp', 'fp', 'fn'):
        axes[0, 1].plot(sweep_df['confidence'], sweep_df[metric], label=metric)
    axes[0, 1].axvline(SELECTED_CONFIDENCE, color='black', linestyle='--')
    axes[0, 1].set_title('Detection counts versus confidence')
    axes[0, 1].legend()
    if roc_auc is not None:
        fpr, tpr, _ = roc_curve(targets, scores)
        axes[1, 0].plot(fpr, tpr, label=f'ROC-AUC={roc_auc:.3f}')
        axes[1, 0].plot([0, 1], [0, 1], 'k--')
        axes[1, 0].legend()
        axes[1, 0].set_title('Image-level ROC-AUC')
    else:
        axes[1, 0].text(0.5, 0.5, 'ROC-AUC unavailable', ha='center', va='center')
    axes[1, 0].set_xlabel('False-positive rate')
    axes[1, 0].set_ylabel('True-positive rate')
    ConfusionMatrixDisplay(confusion_matrix(targets, predicted, labels=[0, 1]), display_labels=['Negative', 'Pith']).plot(ax=axes[1, 1], cmap='Blues', colorbar=False)
    axes[1, 1].set_title('Image-level confusion matrix')
    figure.tight_layout()
    figure.savefig(visual_dir / 'detection_visual_metrics.png', dpi=180)
    plt.close(figure)

    categories = {'tp': [], 'fp': [], 'fn': []}
    for row in visual_rows:
        predictions = [item for item in row['predictions'] if item[4] >= SELECTED_CONFIDENCE]
        match = match_detections([item[:5] for item in predictions], row['truth'])
        categories['fn' if match['fn'] else ('fp' if match['fp'] else 'tp')].append((row, predictions))
    for category, examples in categories.items():
        for index, (row, predictions) in enumerate(examples[:12]):
            with Image.open(row['image_path']) as source:
                image = source.convert('RGB').copy()
            draw = ImageDraw.Draw(image)
            for x1, y1, x2, y2 in row['truth']:
                draw.rectangle((x1, y1, x2, y2), outline='lime', width=4)
            for x1, y1, x2, y2, confidence, _ in predictions:
                draw.rectangle((x1, y1, x2, y2), outline='red', width=4)
                draw.text((x1, max(0, y1 - 18)), f'{confidence:.2f}', fill='red')
            image.save(visual_dir / f'{category}_{index:02d}_{row["image_id"]}.png')
    write_json(visual_dir / 'visual_review_summary.json', {'image_level_roc_auc': roc_auc, 'image_level_accuracy': float(np.mean(np.asarray(targets) == np.asarray(predicted))), 'review_counts': {key: len(value) for key, value in categories.items()}})
    del visual_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()